# Final Evaluation

The detailed comments explaining the pipeline **line by line** are provided in `notebooks/hparam-search.ipynb`.

This notebook serves a different purpose: to **fully exploit and stress-test the parameters of the best trial** obtained from the hyperparameter search.

Therefore, we do not repeat exhaustive inline documentation here. We only comment on points that are important to highlight for final evaluation (key decisions, relevant adjustments, and result interpretation).

# Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
import pandas as pd
import random

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import build_price_basis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter, BlockBootstrapSampler
from src.nn.loss.elasticity_mask import elasticity_entry_mask

# Seeds

In [3]:
BASE_SEED = 42

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Seetings

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = RESULTS_DIR / "checkpoints" / "final_eval_nested"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

NESTED_DIR = RESULTS_DIR / "nested"

# ── Data / Evaluation ─────────────────────────────────────────────
N_UPCS = 5
SMOOTH_WINDOW = 8
BETA_EDA = -2
K_NEIGHBORS = 5  # aligned with hparam-search (was 4)

PROTOCOL = "nested_temporal"
TRAIN_FRAC = 0.8
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
EVAL_SEEDS = [11, 29, 42, 77, 123]
N_BOOTSTRAP = 20
BLOCK_SIZE = 4
MIN_TRAIN_FRAC = 0.5

# ── Training ───────────────────────────────────────────────────────
N_EPOCHS_P0 = 350
N_EPOCHS_P1 = 400
PATIENCE    = 30
ES_PATIENCE = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

with open(NESTED_DIR / "icdn_outer_best_params.json", "r", encoding="utf-8") as f:
    outer_best = json.load(f)
with open(NESTED_DIR / "icdn_holdout_best_params.json", "r", encoding="utf-8") as f:
    holdout_best = json.load(f)

print("Loaded nested outer params:")
print(json.dumps(outer_best, indent=2, ensure_ascii=False))
print("Loaded nested holdout params:")
print(json.dumps(holdout_best, indent=2, ensure_ascii=False))

# Loader

In [ ]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()

_, store_cats = encoder.factorize(df, "store_code",        sort=True)
_, week_cats  = encoder.factorize(df, "week_id",           sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm", sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm",sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)

brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

print(f"Stores: {n_stores}  |  Weeks: {n_weeks}")

mp_builder = MultiProductBuilder()
sorted_weeks = sorted(df["week_id"].unique())
n_sample = max(1, int(len(sorted_weeks) * MIN_TRAIN_FRAC))
sample_weeks = sorted_weeks[:n_sample]
mp_builder.fit_panel(df[df["week_id"].isin(sample_weeks)], n_upcs=N_UPCS)
n_upcs = mp_builder.n
upc_names = mp_builder.selected_upcs

print(f"Selection window: {n_sample} weeks (never used as outer val)")
print(f"UPCs selected: {list(upc_names)}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302
Full wide shape: (19808, 171)
UPCs selected: 5
Top 5: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbors

In [6]:
# Static Metadata per UPC position (same for all batches)
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm", "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

# category_code can be string → factorize for comparability
cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand": torch.tensor(upc_meta["brand_family_norm"].values, dtype=torch.long, device=device),
    "style": torch.tensor(upc_meta["style_segment_norm"].values, dtype=torch.long, device=device),
    "liters": torch.tensor(upc_meta["liters_per_upc"].values, dtype=torch.float32, device=device),
}

# Temporal Folds

In [ ]:
splitter = TemporalSplitter(week_col="week_id")

nested_plans = splitter.nested_expanding_splits(
    df=df,
    n_outer=N_OUTER_FOLDS,
    n_inner=N_INNER_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

def materialize(plan_or_pair):
    if isinstance(plan_or_pair, dict):
        tr_w, va_w = mp_builder.make_fold_frames(plan_or_pair["outer_train"], plan_or_pair["outer_val"])
        plan_or_pair["outer_train"], plan_or_pair["outer_val"] = tr_w, va_w
        plan_or_pair["inner_splits"] = [
            mp_builder.make_fold_frames(tr, va) for tr, va in plan_or_pair["inner_splits"]
        ]
        return plan_or_pair
    tr, va = plan_or_pair
    return mp_builder.make_fold_frames(tr, va)

nested_plans = [materialize(p) for p in nested_plans]

train_final_long, val_final_long = splitter.single_split(df, train_frac=TRAIN_FRAC)
holdout_inner_long = splitter.expanding_splits(
    train_final_long, n_folds=N_INNER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
holdout_weeks = set(val_final_long["week_id"].unique())
for _, inner_val in holdout_inner_long:
    leak = set(inner_val["week_id"].unique()) & holdout_weeks
    if leak:
        raise RuntimeError(f"Holdout leak: {sorted(leak)[:10]}")

holdout_inner = [mp_builder.make_fold_frames(tr, va) for tr, va in holdout_inner_long]
train_final, val_final = mp_builder.make_fold_frames(train_final_long, val_final_long)
train_weeks_final = sorted(train_final["week_id"].unique())

# Functions

In [ ]:
def apply_icdn_params(params: dict) -> None:
    global N_BASIS, HIDDEN, ACT, DROPOUT, D_STORE, D_BRAND, D_STYLE
    global BATCH_SIZE, LR_P0, LR_P1, LAMBDA_SMOOTH, LAMBDA_ELAST
    N_BASIS = int(params["N_BASIS"])
    HIDDEN = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    ACT = "gelu"
    DROPOUT = float(params["DROPOUT"])
    D_STORE = 16
    D_BRAND = 8
    D_STYLE = 8
    BATCH_SIZE = int(params["BATCH_SIZE"])
    LR_P0 = float(params["LR_P0"])
    LR_P1 = float(params["LR_P1"])
    LAMBDA_SMOOTH = float(params["LAMBDA_SMOOTH"])
    LAMBDA_ELAST = float(params["LAMBDA_ELAST"])

In [ ]:
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


def attach_smooth_targets(wide: pd.DataFrame, smooth_window: int = SMOOTH_WINDOW) -> pd.DataFrame:
    """Masked rolling on the chronological unique-week sequence.

    Zeros from fillna in the pivot never enter the mean: unobserved demand
    is NaN, and pandas rolling.mean skips NaN.
    Call this once on the original train/val, never on a bootstrap sample.
    """
    out = wide.sort_values(["store_code", "week_id"]).copy()
    for i in range(n_upcs):
        y = out[f"log_liters_{i}"].where(out[f"obs_mask_{i}"].eq(1))
        out[f"log_liters_smooth_{i}"] = (
            y.groupby(out["store_code"])
             .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
        )
    return out


def _phase0_frame(wide: pd.DataFrame) -> pd.DataFrame:
    """Phase-0 copy: replace log_liters with the precomputed smooth target."""
    s = wide.copy()
    for i in range(n_upcs):
        s[f"log_liters_{i}"] = s[f"log_liters_smooth_{i}"].fillna(0.0)
    return s


def prepare_fold_frames(
    train_wide: pd.DataFrame,
    val_wide: pd.DataFrame,
    recompute_smooth: bool = True,
):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    if recompute_smooth:
        train_wide = attach_smooth_targets(train_wide)
        val_wide   = attach_smooth_targets(val_wide)

    train_wide_s = _phase0_frame(train_wide)
    val_wide_s   = _phase0_frame(val_wide)
    return train_wide, val_wide, train_wide_s, val_wide_s


def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s):
    loader_factory = DataLoaderFactory(num_workers=4, pin_memory=True, persistent_workers=True)

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader_p0 = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=BATCH_SIZE, shuffle=False
    )

    train_loader = loader_factory.create_train_loader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader = loader_factory.create_eval_loader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False
    )

    return train_loader_p0, val_loader_p0, train_loader, val_loader

In [ ]:
# This function is new compared to the notebook hparam-search.ipynb
# because we need to build the model components in a function.
# We encapsulate the model components in an unique function.
def build_model_components(train_wide: pd.DataFrame):
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        obs = train_wide[f"price_observed_{i}"].astype(bool)
        x_i = train_wide.loc[obs, f"log_price_{i}"].values
        if len(x_i) < 10:
            raise ValueError(f"Too few observed prices for UPC {i} to build splines")
        config = builder.build_from_data(
            x_i, n_basis=N_BASIS, q_min=0.05, q_max=0.95, basis_type="truncated_cubic"
        )
        spline_configs.append(config)
        
    price_splines = build_price_basis("truncated_cubic", spline_configs)

    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores,
        d_store=D_STORE,
        n_brands=n_brands,
        d_brand=D_BRAND,
        n_styles=n_styles,
        d_style=D_STYLE,
    )

    def make_model(enforce_negative_beta, use_cross, device):
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=N_BASIS,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=HIDDEN,
            act=ACT,
            dropout=DROPOUT,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        model = ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)
        return model

    return make_model

In [ ]:
def _empty_loss_acc():
    return dict(fit_num=0.0, fit_den=0.0, sm_num=0.0, sm_den=0.0, el_num=0.0, el_den=0.0)

def _accum_loss(acc, logs):
    n_fit = logs["n_obs"].item()
    n_sm  = logs["n_smooth"].item()
    n_el  = logs["n_elast"].item()
    acc["fit_num"] += logs["loss_fit"].item()    * n_fit
    acc["fit_den"] += n_fit
    acc["sm_num"]  += logs["loss_smooth"].item() * n_sm
    acc["sm_den"]  += n_sm
    acc["el_num"]  += logs["loss_elast"].item()  * n_el
    acc["el_den"]  += n_el

def _epoch_loss(acc, loss_fn):
    return (
        acc["fit_num"] / max(acc["fit_den"], 1.0)
        + loss_fn.lambda_smooth * (acc["sm_num"] / max(acc["sm_den"], 1.0))
        + loss_fn.lambda_elast  * (acc["el_num"] / max(acc["el_den"], 1.0))
    )

In [ ]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name=""):
    """val_loader is None → fixed-length train, last-epoch checkpoint, no ES."""
    use_val = val_loader is not None
    best_val_loss = float("inf")
    best_epoch    = n_epochs
    no_improve    = 0
    history       = {"train_loss": [], "val_loss": [], "val_mae": [], "lr": []}
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    print(f"\n{'='*70}")
    print(f"  {phase_name}  |  {n_epochs} epochs  |  ckpt: {ckpt_path.name}"
          f"  |  val={'on' if use_val else 'off'}")
    print(f"{'='*70}")

    for epoch in range(n_epochs):
        model.train()
        train_acc = _empty_loss_acc()

        for batch in train_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                        aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                        availability=batch["availability"], price_observed=batch["price_observed"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                    aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                    availability=batch["availability"], price_observed=batch["price_observed"])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            _accum_loss(train_acc, logs)

        train_loss = _epoch_loss(train_acc, loss_fn)
        history["train_loss"].append(train_loss)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if not use_val:
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1:4d}/{n_epochs}"
                      f"  train={train_loss:.4f}")
            continue

        model.eval()
        val_acc = _empty_loss_acc()
        total_abs, total_mask = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true   = batch["demands"]
                obs_mask = batch["obs_mask"]

                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask,
                                aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                availability=batch["availability"], price_observed=batch["price_observed"])

                _accum_loss(val_acc, logs)
                total_abs  += ((y_hat - y_true).abs() * obs_mask).sum().item()
                total_mask += obs_mask.sum().item()

        val_loss = _epoch_loss(val_acc, loss_fn)
        val_mae  = total_abs / max(total_mask, 1.0)

        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)

        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch + 1
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
            saved = "x"
        else:
            no_improve += 1
            saved = ""

        if (epoch + 1) % 10 == 0 or no_improve == 0:
            print(f"Epoch {epoch+1:4d}/{n_epochs}"
                  f"  train={train_loss:.4f}"
                  f"  val={val_loss:.4f}"
                  f"  mae={val_mae:.4f}"
                  f"  {saved}")

        if no_improve >= es_patience:
            print(f"  Early stopping in epoch {epoch+1}")
            break

    if not use_val:
        torch.save(model.state_dict(), ckpt_path)
        print(f"\nLast-epoch checkpoint: {ckpt_path.name}  epoch={best_epoch}")
    else:
        print(f"\nBest val_loss: {best_val_loss:.4f}  epoch={best_epoch} - {ckpt_path.name}")

    return {
        "best_epoch": best_epoch,
        "best_val_loss": None if not use_val else best_val_loss,
        "history": history,
    }

In [11]:
# We add these helpers because they are often used in the
# execution of the notebook. In fact, we could have added them
# to the hparam-search notebook too.

def zero_and_freeze_nonlinear(model):
    ph = model.head.param_head
    for attr in ("head_w", "head_w_cross", "head_cross"):
        if not hasattr(ph, attr):
            continue
        layer = getattr(ph, attr)
        with torch.no_grad():
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()
        layer.weight.requires_grad_(False)
        if layer.bias is not None:
            layer.bias.requires_grad_(False)
        
def unfreeze_nonlinear(model):
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(True)
        head.bias.requires_grad_(True)

def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0
    )
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)
    print(f"beta initialized: target={beta_target:.3f}  beta_raw_init={beta_raw_init:.4f}")

print("Helpers defined")

Helpers defined


In [ ]:
def compute_global_metrics(model, val_loader):
    model.eval()
    all_y_hat, all_y_true, all_mask = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]  

            all_y_hat.append(y_hat.cpu())
            all_y_true.append(y_true.cpu())
            all_mask.append(obs_mask.cpu())

    y_hat = torch.cat(all_y_hat)
    y_true = torch.cat(all_y_true)
    mask = torch.cat(all_mask)

    mae = float(((y_hat - y_true).abs() * mask).sum() / mask.sum())
    rmse = float(torch.sqrt((((y_hat - y_true) ** 2) * mask).sum() / mask.sum()))

    preds_np = y_hat.numpy()
    targets_np = y_true.numpy()
    mask_np = mask.bool().numpy()

    ss_res = ((targets_np[mask_np] - preds_np[mask_np]) ** 2).sum()
    ss_tot = ((targets_np[mask_np] - targets_np[mask_np].mean()) ** 2).sum()
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }

# This function extract the elasticity rows from the model.
def extract_elasticity_rows(model, val_loader, run_type, run_id, fold=None, seed=None, bootstrap_run=None):
    model.eval() # We set the model to evaluation mode.
    rows = []

    with torch.no_grad(): # We don't need to compute the gradients.
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            # ids[:, 0] holds the store_code (label-encoded contiguous index).
            store_idx  = batch["ids"][:, 0].cpu().numpy()
            # We recover the store codes from the store indices.
            # Recall that they were encoded as integers.
            # Namely, [101, 123, 103] <-> [0, 1, 2]
            store_code = np.array(store_cats)[store_idx]
            # obs_mask is already (B, n) — pre-stacked in MultiProductDataset.__init__
            y_hat, _, aux = model.run(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            E = aux["E"].cpu().numpy()
            # demands is already (B, n) — pre-stacked in MultiProductDataset.__init__
            y_true_np = batch["demands"].cpu().numpy()
            y_hat_np = y_hat.cpu().numpy()

            # Elasticity tensor shape: (B, n_upcs, n_upcs)
            B = E.shape[0]
            # We recover the UPC names. We extract them from the 
            # MultiProductBuilder object.
            upc_names = np.array(mp_builder.selected_upcs)

            M = elasticity_entry_mask(
                batch["obs_mask"],
                pairs=aux["pairs"],
                availability=batch["availability"],
                price_observed=batch["price_observed"],
            ).cpu().numpy()

            for b in range(B):
                sc = store_code[b]
                for i in range(n_upcs):
                    for j in range(n_upcs):
                        if not M[b, i, j]:
                            continue
                        rows.append({
                            "run_type": run_type,
                            "run_id": run_id,
                            "fold": fold,
                            "seed": seed,
                            "bootstrap_run": bootstrap_run,
                            "store_code": sc,
                            "upc_i": upc_names[i],
                            "upc_j": upc_names[j],
                            "type": "own" if i == j else "cross",
                            "E": E[b, i, j],
                            "y_true_i": y_true_np[b, i],
                            "y_hat_i": y_hat_np[b, i],
                        })
    return pd.DataFrame(rows)

In [ ]:
def train_one_run(
    train_fold,
    eval_fold,
    seed,
    run_type,
    run_id,
    n_epochs_p0,
    n_epochs_p1,
    fold=None,
    bootstrap_run=None,
    recompute_smooth=True,
):
    set_all_seeds(seed)
    n_epochs_p0 = int(n_epochs_p0)
    n_epochs_p1 = int(n_epochs_p1)
    train_wide, val_wide, train_wide_s, val_wide_s = prepare_fold_frames(
        train_fold, eval_fold, recompute_smooth=recompute_smooth,
    )
    # eval loaders se usan SOLO al final. No entran en run_training.
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s
    )

    # We build the model components.
    make_model = build_model_components(train_wide)

    # We define the checkpoint paths.
    ckpt_p0 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase1.pt"

    # ── PHASE 0 ─────────────────────────────────────────────────────
    # The goal of this phase is to obtain a robust initialization before
    # unlocking the model's full flexibility. To do so:
    #
    #   1. Cross-price effects are available (use_cross=True) and the spline
    #      weights are frozen (head_w → zeros, requires_grad=False). This reduces
    #      the model to a log-linear demand:
    #      log(q_i) \approx b + beta·log(p_i) + sum_{j\neq i}·alpha_{ij}·log(p_j) log(p_i) .
    #
    #   2. The head_beta bias is initialized with the inverse softplus of
    #      BETA_EDA, so that the own-price elasticity at startup equals exactly
    #      -BETA_EDA. This gives the model an economically sensible starting
    #      point instead of a random one.
    #
    #   3. The loss applies no smoothness or positivity penalties (lambda_smooth=0,
    #      lambda_pos=0): only the demand prediction error is minimized.
    #
    # By the end of this phase, beta and b are well calibrated, which makes
    # convergence easier in later phases when spline weights and cross-price
    # effects are unfrozen.

    model_p0 = make_model(enforce_negative_beta=True, use_cross=True, device=device)

    # We freeze the spline weights.
    zero_and_freeze_nonlinear(model_p0)

    # We initialize the beta bias.
    init_beta_prior(model_p0, BETA_EDA)
    with torch.no_grad():
        model_p0.head.param_head.head_beta_cross.weight.zero_()
        model_p0.head.param_head.head_beta_cross.bias.zero_()

    # We define the loss function.
    loss_p0 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=LAMBDA_SMOOTH,
        lambda_elast=LAMBDA_ELAST,
        reduction="mean"
    )
    decay, no_decay = [], []
    for name, p in model_p0.named_parameters():
        if not p.requires_grad: continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P0,
    )
    sch_p0 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_p0, T_max=max(n_epochs_p0, 1), eta_min=1e-5
    )

    run_training(
        model=model_p0,
        train_loader=train_loader_p0,
        val_loader=None,
        loss_fn=loss_p0,
        optimizer=opt_p0,
        scheduler=sch_p0,
        n_epochs=n_epochs_p0,
        es_patience=n_epochs_p0,
        ckpt_path=ckpt_p0,
        device=device,
        neighbor_meta=neighbor_meta,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P0",
    )

    # ── Phase 1: Unlock spline weights with smoothed targets ───────────────────
    # Building on the stable beta and b from Phase 0, this phase introduces the
    # spline flexibility that was previously frozen:
    #
    #   1. The model is initialized from the Phase 0 checkpoint. The spline
    #      weights (head_w) are unfrozen (requires_grad=True), allowing the
    #      model to learn non-linear price responses beyond the log-linear baseline.
    #
    #   2. The loss applies smoothness or positivity penalties
    #      (lambda_smooth\neq0, lambda_pos\neq0): pure fit to the training data.
    #
    #   3. Training uses the non-smoothed data (train_loader).
    #      eval_fold / val_loader se usan solo DESPUÉS, para métricas.
    #
    # By the end of this phase, the spline shapes are well fit to the raw demand
    # signal, providing a good initialization for the cross-price phase that follows.

    model_p1 = make_model(enforce_negative_beta=True, use_cross=True, device=device)
    model_p1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    unfreeze_nonlinear(model_p1)

    loss_p1 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=LAMBDA_SMOOTH,
        lambda_elast=LAMBDA_ELAST,
        reduction="mean"
    )
    decay, no_decay = [], []
    for name, p in model_p1.named_parameters():
        if not p.requires_grad: continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P1,
    )
    sch_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_p1, T_max=max(n_epochs_p1, 1), eta_min=1e-5
    )

    run_training(
        model=model_p1,
        train_loader=train_loader,
        val_loader=None,
        loss_fn=loss_p1,
        optimizer=opt_p1,
        scheduler=sch_p1,
        n_epochs=n_epochs_p1,
        es_patience=n_epochs_p1,
        ckpt_path=ckpt_p1,
        device=device,
        neighbor_meta=neighbor_meta,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P1",
    )

    # Last-epoch weights already saved by run_training (no outer-val checkpointing).
    model_p1.load_state_dict(torch.load(ckpt_p1, map_location=device))

    # Freeze P* once on the converged model. From this point on,
    # run() uses the O(B * E * d_attn) sparse path.
    model_p1.eval()
    selector = model_p1.head.neighbor_selector
    def h_iter(loader):
        with torch.no_grad():
            for batch in loader:
                batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                tokens = model_p1.context_builder(batch)
                h      = model_p1.head.encoder(tokens)
                yield h
    global_mean = selector.accumulate_mean_scores(
        h_iter(train_loader),
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )
    selector.freeze_graph(
        global_mean,
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )

    metrics = compute_global_metrics(model_p1, val_loader)
    df_e = extract_elasticity_rows(
        model=model_p1,
        val_loader=val_loader,
        run_type=run_type,
        run_id=run_id,
        fold=fold,
        seed=seed,
        bootstrap_run=bootstrap_run,
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return metrics, df_e

# Training - Folds and Seeds

In [ ]:
params_by_outer = {int(p["outer_id"]): p for p in outer_best}

fold_metrics_rows = []
fold_elasticity_rows = []

for plan in nested_plans:
    fold_id = plan["outer_id"]
    payload = params_by_outer[fold_id]
    apply_icdn_params(payload["params"])
    print(
        f"\n=== Outer {fold_id} | selected trial {payload['trial']} "
        f"| ep0={payload['best_epoch_p0']} ep1={payload['best_epoch_p1']} ==="
    )

    for seed in EVAL_SEEDS:
        print(f"\n=== Fold {fold_id} | Seed {seed} ===")

        metrics, df_e = train_one_run(
            train_fold=plan["outer_train"],
            eval_fold=plan["outer_val"],
            seed=seed,
            run_type="kfold_nested",
            run_id=f"fold{fold_id}_seed{seed}",
            n_epochs_p0=payload["best_epoch_p0"],
            n_epochs_p1=payload["best_epoch_p1"],
            fold=fold_id,
            bootstrap_run=None,
        )

        fold_metrics_rows.append({
            "protocol": PROTOCOL,
            "fold": fold_id,
            "seed": seed,
            "n_train": len(plan["outer_train"]),
            "n_val": len(plan["outer_val"]),
            "selected_trial": payload["trial"],
            "robust_score_inner": payload["robust_score"],
            "best_epoch_p0": payload["best_epoch_p0"],
            "best_epoch_p1": payload["best_epoch_p1"],
            **metrics,
        })
        df_e = df_e.copy()
        df_e["protocol"] = PROTOCOL
        fold_elasticity_rows.append(df_e)

nn_kfold_metrics_raw = pd.DataFrame(fold_metrics_rows)
nn_kfold_elasticities_raw = pd.concat(fold_elasticity_rows, ignore_index=True)

print("K-fold + seeds completed")

# Fold - Summary

In [15]:
nn_kfold_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_kfold_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_kfold_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_kfold_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_kfold_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_kfold_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_kfold_metrics_raw["r2_val"].std(ddof=1),
    "n_runs": len(nn_kfold_metrics_raw),
}])

display(nn_kfold_metrics_summary)

,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std,n_runs
0,0.467724,0.015977,0.603901,0.024591,0.649631,0.112975,25


In [16]:
df_e = nn_kfold_elasticities_raw.copy()
# --- Own elasticities ---
own_elast_summary = (
    df_e[df_e["type"] == "own"]
    .groupby(["store_code", "upc_i"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", lambda s: s.quantile(0.025)),
        elasticity_ci_high=("E", lambda s: s.quantile(0.975)),
        n_obs=("E", "size"),
    )
    .rename(columns={"upc_i": "upc_code"})
    .sort_values(["store_code", "upc_code"])
    .reset_index(drop=True)
)
# --- Cross elasticities ---
cross_elast_summary = (
    df_e[df_e["type"] == "cross"]
    .groupby(["store_code", "upc_i", "upc_j"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", lambda s: s.quantile(0.025)),
        elasticity_ci_high=("E", lambda s: s.quantile(0.975)),
        n_obs=("E", "size"),
    )
    .sort_values(["store_code", "upc_i", "upc_j"])
    .reset_index(drop=True)
)
display(own_elast_summary.head())
display(cross_elast_summary.head())

,store_code,upc_code,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,3410017306,-1.814122,0.859542,-3.367340,-0.425288,345
1,5,7289000011,-1.251867,0.456814,-2.203219,-0.398948,440
2,8,1820000784,-2.284638,1.065392,-4.788553,-1.016560,750
3,8,3410010505,-1.502022,0.703030,-4.426843,-0.859423,750
4,8,3410017306,-2.342446,1.203361,-4.450488,-0.016593,345


,store_code,upc_i,upc_j,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,3410017306,7289000011,0.336160,0.331640,-0.303397,0.880717,280
1,5,7289000011,3410017306,0.164843,0.342657,-0.596347,0.913305,280
2,8,1820000784,3410010505,0.629952,0.236214,-0.137039,1.059660,750
3,8,1820000784,3410017306,0.115934,0.723655,-1.150006,2.153180,345
4,8,1820000784,7289000011,0.313906,0.178482,-0.000531,0.665050,695


# Training - Bootstrapping

In [ ]:
# Smooth phase-0 targets once on the original chronological train/val.
# Bootstrap will resample these rows; it must not recompute the rolling.
train_final_p0 = attach_smooth_targets(train_final)
val_final_p0   = attach_smooth_targets(val_final)
train_weeks_final = sorted(train_final_p0["week_id"].unique())

print(f"Train final: {len(train_final_p0):,} rows | {len(train_weeks_final)} weeks")
print(f"Val final:   {len(val_final_p0):,} rows | {val_final_p0['week_id'].nunique()} weeks")

Train final: 15,822 rows | 241 weeks
Val final:   3,986 rows | 61 weeks


In [18]:
# We create the bootstrap sampler. 
# To visualize:
# ─────────────────────────────────────────────────────────────────────────────
# Suppose we have 10 training weeks and block_size=4.
#
#   train_weeks  = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
#   block_starts = [0, 4, 8]   (non-overlapping block start indices)
#   n_blocks     = 3
#
# We draw n_blocks indices WITH replacement:
#   sampled_indices = [2, 0, 2]    (block 2 drawn twice, block 1 never drawn)
#
#   idx=2  start=8: weeks [9, 10]       (short tail block, < block_size)
#   idx=0  start=0: weeks [1, 2, 3, 4]
#   idx=2  start=8: weeks [9, 10]       (repeated)
#
# Bootstrap sample contains weeks: [9,10,1,2,3,4, 9,10]
#   · Weeks 5-8 are absent. The model never sees them in this run.
#   · Weeks 9-10 appear twice. Their observations are counted double.
#
# Repeating this N_BOOTSTRAP times gives N_BOOTSTRAP slightly different
# training sets, producing a distribution of model outputs from which
# we can estimate uncertainty (confidence intervals, std of elasticities).
# ─────────────────────────────────────────────────────────────────────────────

bootstrap_sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=BLOCK_SIZE,
    rng=np.random.default_rng(BASE_SEED),
)

In [ ]:
apply_icdn_params(holdout_best["params"])
BOOTSTRAP_TRAINING_SEED = EVAL_SEEDS[0]

bootstrap_metrics_rows = []
bootstrap_elasticity_rows = []

print(
    f"Holdout fixed epochs: P0={holdout_best['best_epoch_p0']} "
    f"P1={holdout_best['best_epoch_p1']}"
)

for b in range(N_BOOTSTRAP):
    print(f"\n=== Bootstrap {b+1}/{N_BOOTSTRAP} ===")

    train_bs = bootstrap_sampler.sample(train_final_p0, train_weeks_final)

    metrics, df_e = train_one_run(
        train_fold=train_bs,
        eval_fold=val_final_p0,
        seed=BOOTSTRAP_TRAINING_SEED,
        run_type="bootstrap_nested",
        run_id=f"bootstrap{b}",
        n_epochs_p0=holdout_best["best_epoch_p0"],
        n_epochs_p1=holdout_best["best_epoch_p1"],
        fold=None,
        bootstrap_run=b,
        recompute_smooth=False,
    )

    bootstrap_metrics_rows.append({
        "protocol": PROTOCOL,
        "bootstrap_run": b,
        "seed": BOOTSTRAP_TRAINING_SEED,
        "n_train": len(train_bs),
        "n_val": len(val_final_p0),
        "selected_trial": holdout_best["trial"],
        "best_epoch_p0": holdout_best["best_epoch_p0"],
        "best_epoch_p1": holdout_best["best_epoch_p1"],
        **metrics,
    })
    df_e = df_e.copy()
    df_e["protocol"] = PROTOCOL
    bootstrap_elasticity_rows.append(df_e)

nn_bootstrap_metrics_raw = pd.DataFrame(bootstrap_metrics_rows)
nn_bootstrap_elasticities_raw = pd.concat(bootstrap_elasticity_rows, ignore_index=True)

print("Bootstrap completed")
display(nn_bootstrap_metrics_raw.head())

# Bootstrapping - Summary

In [20]:
# Defining the quantile functions.
def q025(x): return np.percentile(x, 2.5)
def q975(x): return np.percentile(x, 97.5)

# Own-elasticity summary.
nn_bootstrap_own_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["type"] == "own"]
    .groupby(["store_code", "upc_i"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
    .rename(columns={"upc_i": "upc_code"})
)

# Cross-elasticity summary.
nn_bootstrap_cross_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["type"] == "cross"]
    .groupby(["store_code", "upc_i", "upc_j"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
)

display(nn_bootstrap_own_summary.head())
display(nn_bootstrap_cross_summary.head())

,store_code,upc_code,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,7289000011,-1.093612,0.430375,-2.053460,-0.282824,280
1,8,1820000784,-2.020780,0.993626,-3.984668,-0.615834,1220
2,8,3410010505,-0.975422,0.369606,-1.649501,-0.238207,1220
3,8,7289000011,-1.138738,0.494822,-2.420387,-0.343515,1220
4,9,1820000784,-1.782275,0.870329,-3.633047,-0.489981,1040


,store_code,upc_i,upc_j,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,8,1820000784,3410010505,0.724869,0.155056,0.419611,0.992044,1220
1,8,1820000784,7289000011,0.405757,0.152675,0.109429,0.696803,1220
2,8,3410010505,1820000784,0.146220,0.549222,-1.083003,0.886491,1220
3,8,3410010505,7289000011,0.326387,0.163150,0.037512,0.628590,1220
4,8,7289000011,1820000784,0.198030,0.382346,-0.566605,0.739863,1220


In [21]:
# Bootstrapping - Metrics Summary
nn_bootstrap_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_bootstrap_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_bootstrap_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_bootstrap_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_bootstrap_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_bootstrap_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_bootstrap_metrics_raw["r2_val"].std(ddof=1),
    "n_bootstrap_runs": len(nn_bootstrap_metrics_raw),
}])

display(nn_bootstrap_metrics_summary)

,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std,n_bootstrap_runs
0,0.493897,0.020432,0.628093,0.018553,0.471428,0.031625,20


# Save Results

In [ ]:
nn_kfold_elasticities_raw.to_csv(DATA_DIR / "nn_kfold_elasticities_raw_nested.csv", index=False)
nn_kfold_metrics_raw.to_csv(DATA_DIR / "nn_kfold_metrics_raw_nested.csv", index=False)
nn_bootstrap_elasticities_raw.to_csv(DATA_DIR / "nn_bootstrap_elasticities_raw_nested.csv", index=False)

print("Saved:")
print("- nn_kfold_elasticities_raw_nested.csv")
print("- nn_kfold_metrics_raw_nested.csv")
print("- nn_bootstrap_elasticities_raw_nested.csv")